# Convert Spatial Z-score Matrices to Long Format

This notebook converts the wide spatial z-score files into the long-format CSVs expected by the website.

Run the cells in order. The output files will be written into the same local folder as the source files by default.

In [ ]:
from pathlib import Path
import csv

DATA_DIR = Path(r"C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles")
RNA_INPUT = DATA_DIR / "rna_zscore.csv"
PROT_INPUT = DATA_DIR / "prot_zscore.csv"
RNA_OUTPUT = DATA_DIR / "rna_zscore_long.csv"
PROT_OUTPUT = DATA_DIR / "prot_zscore_long.csv"

print("Input folder:", DATA_DIR)
print("RNA input exists:", RNA_INPUT.exists(), RNA_INPUT)
print("Protein input exists:", PROT_INPUT.exists(), PROT_INPUT)
print("RNA output:", RNA_OUTPUT)
print("Protein output:", PROT_OUTPUT)

In [ ]:
GROUP_BY_PREFIX = {
    "P": "Posterior",
    "A": "Anterior",
    "S": "Somite",
}

ROMAN_TO_INT = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
}

def infer_group(sample_name: str) -> str:
    sample_name = (sample_name or "").strip()
    if not sample_name:
        return ""
    return GROUP_BY_PREFIX.get(sample_name[0].upper(), "")

def infer_replicate(sample_name: str) -> str:
    sample_name = (sample_name or "").strip()
    if not sample_name or "-" not in sample_name:
        return ""

    suffix = sample_name.split("-", 1)[1].strip().upper()
    if suffix in ROMAN_TO_INT:
        return str(ROMAN_TO_INT[suffix])
    if suffix.isdigit():
        return suffix
    return ""

def convert_wide_to_long(input_path: Path, output_path: Path) -> int:
    with input_path.open("r", newline="", encoding="utf-8-sig") as infile:
        reader = csv.DictReader(infile)
        if not reader.fieldnames:
            raise ValueError(f"No header found in {input_path}")

        sample_field = None
        for candidate in ("sample", "Sample"):
            if candidate in reader.fieldnames:
                sample_field = candidate
                break

        if sample_field is None:
            raise ValueError(
                f"Expected a 'sample' column in {input_path}, found: {', '.join(reader.fieldnames[:10])}"
            )

        gene_fields = [field for field in reader.fieldnames if field != sample_field]
        output_path.parent.mkdir(parents=True, exist_ok=True)

        row_count = 0
        with output_path.open("w", newline="", encoding="utf-8") as outfile:
            writer = csv.DictWriter(
                outfile,
                fieldnames=["sample", "Gene", "Z-score", "group", "replicate"],
            )
            writer.writeheader()

            for row in reader:
                sample_name = (row.get(sample_field) or "").strip()
                group = infer_group(sample_name)
                replicate = infer_replicate(sample_name)

                for gene_name in gene_fields:
                    value = row.get(gene_name)
                    if value in (None, ""):
                        continue

                    writer.writerow({
                        "sample": sample_name,
                        "Gene": gene_name,
                        "Z-score": value,
                        "group": group,
                        "replicate": replicate,
                    })
                    row_count += 1

        return row_count

In [ ]:
rna_rows = convert_wide_to_long(RNA_INPUT, RNA_OUTPUT)
prot_rows = convert_wide_to_long(PROT_INPUT, PROT_OUTPUT)

print(f"Wrote {rna_rows:,} RNA rows to {RNA_OUTPUT}")
print(f"Wrote {prot_rows:,} protein rows to {PROT_OUTPUT}")

In [ ]:
def preview_csv(path: Path, limit: int = 5):
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            print(line.rstrip())
            if index + 1 >= limit:
                break

print("RNA preview:")
preview_csv(RNA_OUTPUT)
print()
print("Protein preview:")
preview_csv(PROT_OUTPUT)

## Next step

After the two long-format CSV files are created, upload them to GitHub. Once they are in the repo or your data repository, the website can be pointed directly at those files instead of reshaping the wide matrices in the browser.